# 03 - Action：执行实验与数据收集

> **STAR法则的第三步**：执行实验，生成模拟数据，验证数据质量。

---

## 📋 学习目标

完成本notebook后，你将能够：
1. 生成符合真实分布的模拟实验数据
2. 验证分组比例和随机化质量
3. 执行AA检验
4. 识别和处理脏数据

---

## 🎲 数据生成

### 生成用户数据

**TODO 1**：生成实验用户数据，包含用户属性、分组、各项指标。

```python
import numpy as np
import pandas as pd
import hashlib

np.random.seed(42)

# 实验参数
n_users = 10000  # 总用户数

# 用户属性
users = pd.DataFrame({
    "user_id": [f"U_{10000+i}" for i in range(n_users)],
    "user_type": np.random.choice(["new", "old"], n_users, p=[0.3, 0.7]),
    "device_type": np.random.choice(["iOS", "Android"], n_users, p=[0.4, 0.6]),
})

# 分组函数（复用02中设计的函数）
def assign_group(user_id, salt="experiment_2024_01", num_buckets=1000):
    hash_input = f"{user_id}_{salt}"
    hash_value = int(hashlib.md5(hash_input.encode()).hexdigest(), 16)
    bucket = hash_value % num_buckets
    return "control" if bucket < num_buckets // 2 else "treatment"

users["group"] = users["user_id"].apply(assign_group)

print("用户数据预览：")
print(users.head(10))
print(f"\n分组比例：\n{users['group'].value_counts()}")
```

### 生成历史数据（用于CUPED）

```python
# 历史停留时长（对数正态分布，模拟真实APP用户行为）
users["historical_dwell_time"] = np.random.lognormal(
    mean=np.log(1500), sigma=0.8, size=n_users
)

# 新用户历史时长偏低（模拟真实场景）
new_user_mask = users["user_type"] == "new"
users.loc[new_user_mask, "historical_dwell_time"] *= 0.6

print("\n历史停留时长分布：")
print(users["historical_dwell_time"].describe())
```

### 生成实验期间数据

**TODO 2**：生成实验期间的各项指标数据。

```python
def generate_experiment_data(df):
    """
    生成实验期间的模拟数据
    
    业务逻辑：
    - 对照组：与历史持平
    - 实验组：新算法提升停留时长
      - 老用户效果更明显（+18%）
      - 新用户效果较弱（+8%）
    - CTR：实验组略降（推荐更精准但曝光减少）
    - 次日留存：实验组略升
    """
    df = df.copy()
    
    # TODO：生成停留时长
    # 提示：基于历史时长，根据分组和用户类型添加效果
    
    # TODO：生成CTR（Beta分布）
    # 提示：对照组base CTR，实验组略降
    
    # TODO：生成次日留存（二项分布）
    # 提示：对照组base retention，实验组略升
    
    return df

users = generate_experiment_data(users)

print("\n实验数据预览：")
print(users[["user_id", "group", "dwell_time", "ctr", "retention_next_day"]].head(10))
```

### 添加脏数据

**TODO 3**：添加真实的脏数据问题。

```python
def add_dirty_data(df):
    """
    添加脏数据，模拟真实场景
    """
    df = df.copy()
    
    # TODO：添加缺失值
    # 约2%的dwell_time缺失
    
    # TODO：添加异常值
    # 约0.5%的停留时长异常高（>2小时）
    
    # TODO：添加重复值
    # 少量重复用户ID
    
    return df

users = add_dirty_data(users)

print("\n脏数据统计：")
print(f"缺失值比例：\n{users.isnull().mean()}")
print(f"\n重复user_id数：{users['user_id'].duplicated().sum()}")
```

---

## 🔍 数据质量验证

### 分组平衡检查

```python
print("分组比例检查：")
group_counts = users["group"].value_counts()
print(group_counts)
print(f"\n比例差异: {abs(group_counts['control'] - group_counts['treatment'])}")

# 按用户类型分层检查
print("\n按用户类型分层：")
print(pd.crosstab(users["user_type"], users["group"], margins=True))

# 卡方检验
from scipy.stats import chi2_contingency

contingency = pd.crosstab(users["user_type"], users["group"])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"\n卡方检验: χ²={chi2:.4f}, p={p:.4f}")
print(f"分组与用户类型是否独立: {'是' if p > 0.05 else '否'}")
```

### AA检验

**TODO 4**：对历史数据执行AA检验。

```python
from scipy import stats

def aa_test(df, metric, group_col="group"):
    """
    AA检验：检验实验前两组在指标上是否平衡
    
    参数:
        df: 数据框
        metric: 检验指标
        group_col: 分组列名
    
    返回:
        bool: 是否通过AA检验
    """
    control = df[df[group_col] == "control"][metric].dropna()
    treatment = df[df[group_col] == "treatment"][metric].dropna()
    
    # TODO：执行t检验
    # 提示：使用scipy.stats.ttest_ind
    
    print(f"AA检验 - {metric}")
    print(f"  对照组: n={len(control)}, mean={control.mean():.2f}, std={control.std():.2f}")
    print(f"  实验组: n={len(treatment)}, mean={treatment.mean():.2f}, std={treatment.std():.2f}")
    print(f"  差异: {treatment.mean() - control.mean():.2f}")
    # TODO：打印t统计量和p值
    print()
    
    # TODO：返回检验结果
    pass

# 执行AA检验
print("="*60)
print("AA检验：历史停留时长")
print("="*60)
aa_pass = aa_test(users, "historical_dwell_time")

print(f"\nAA检验结果: {'通过 ✓' if aa_pass else '未通过 ✗'}")

if not aa_pass:
    print("\n警告：AA检验未通过！")
    print("可能原因：")
    print("1. 随机化有问题")
    print("2. 分组前数据已经不平衡")
    print("3. 样本量不足")
```

---

## 💾 保存数据

```python
# 保存原始数据
users.to_csv("../data/raw/experiment_data.csv", index=False)

print("数据已保存到 data/raw/experiment_data.csv")
print(f"\n数据形状: {users.shape}")
print(f"列: {list(users.columns)}")
```

---

## 📝 本节总结

完成本节，你应该已经：

- [ ] 生成符合真实分布的模拟数据
- [ ] 添加脏数据（缺失值、异常值、重复值）
- [ ] 验证分组比例和随机化质量
- [ ] 执行AA检验
- [ ] 保存原始数据

**下一步**：进入 `04-action-analysis.ipynb`，清洗数据并执行分析。

---

> 💡 **关键洞察**：
> 数据质量是分析的前提。
> AA检验不通过，后续分析都不可信。
